# 05 — Baseline Models

**Pipeline role.** Fifth notebook. Trains a roster of untuned classical classifiers on `feature_set_v1` using a fixed evaluation protocol so that later tuning improvements in notebook 06 are comparable.

**Rubric coverage (from `Group Project Guideline.md`).**
- Report Section 3 — Model Building: multiple data mining techniques, with parameter values and conclusions per model.
- Report Section 4 — Performance Evaluation: sets up the comparison table the report will cite.

**TL;DR for teammates.** Same features, same split, same metrics, same `random_state` for every baseline. The purpose here is honest comparison, not maximum performance. Tuning lives in notebook 06.


## Inputs, Outputs, Artifacts

**Inputs.**
- `data/processed/X_train_feat_v1.*`, `y_train_v1.*` from notebook 04.
- Validation protocol conventions from [../references/project_conventions.md](../references/project_conventions.md).

**Outputs.**
- `reports/tables/baseline_results_v1.csv` — per-model metrics from the same validation protocol.
- `reports/figures/baseline_metric_comparison.png` — visual comparison of models on the primary metric.
- Appended rows in [../reports/tables/experiment_log_template.csv](../reports/tables/experiment_log_template.csv) (or working copy), one row per baseline.
- A shortlist of 2–3 models that go into notebook 06 for tuning.

**Downstream consumers.**
- Notebook 06 picks the shortlist to tune.
- Notebook 07 compares final tuned performance against the baseline table.
- Notebook 08 reproduces the comparison table for the report.


## Methodological Influences

Baseline evaluation follows standard ISOM3360 classification practice: stratified k-fold cross-validation, ROC-AUC as the primary threshold-invariant metric, and explicit comparison tables. See the shared [project conventions](../references/project_conventions.md) for standards applied across all notebooks.


## Key Questions To Answer Here

1. What is the floor performance (majority-class / dummy classifier)?
2. Which untuned classical models clearly beat that floor?
3. How do the models compare on ROC-AUC, F1, precision, recall, and accuracy?
4. Which 2–3 models are worth promoting to notebook 06 for tuning, and why?
5. Are there any obvious diagnostics (e.g., severe imbalance effects) that should change the tuning protocol?


## Work Plan

### 5.1 Setup
- Load `X_train_feat_v1`, `y_train_v1`.
- Import baseline estimators from scikit-learn.
- Fix `random_state=42` globally.

### 5.2 Evaluation Protocol (Fixed For All Baselines)
- Stratified k-fold cross-validation on the training set (e.g., `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`).
- Metrics per fold: ROC-AUC (primary), F1, precision, recall, accuracy.
- Report mean ± std across folds.
- Decision threshold for F1/precision/recall at this stage: default `0.5`. Threshold tuning belongs to notebook 06.

### 5.3 Baseline Model Roster
Interpretable classical classifiers chosen for both predictive adequacy and business readability — revenue and operations reviewers can read coefficients, tree splits, or neighbourhood rules without specialized training:

- `DummyClassifier(strategy="most_frequent")` — mandatory floor baseline.
- `LogisticRegression` (scaled features).
- `DecisionTreeClassifier` (default depth).
- `RandomForestClassifier` (modest `n_estimators`, e.g., 100).
- `GradientBoostingClassifier` or `HistGradientBoostingClassifier` — one representative.
- `KNeighborsClassifier` — one scale-sensitive neighbour model (optional but useful for contrast).
- `GaussianNB` or `BernoulliNB` — probabilistic baseline (optional).

### 5.4 Run And Record
- For each model, fit on each fold and record metrics.
- Assemble a long-form results table with columns: `model_name`, `metric`, `mean`, `std`, `notes`.
- Pivot to a wide comparison table for the report.

### 5.5 Interpretation
- Which models beat the majority-class baseline, and by how much?
- Which metric patterns stand out (e.g., high recall but low precision)?
- Which 2–3 models go forward into tuning? Justify on a mix of (a) performance, (b) interpretability, (c) tractability of tuning.

### 5.6 Persist Artifacts
- Save results to `reports/tables/baseline_results_v1.csv`.
- Save comparison figure to `reports/figures/baseline_metric_comparison.png`.
- Append one row per baseline to the experiment log.


## Implementation Block 5.1 — Setup

**Scope.** Section 5.1 only — load the frozen `feature_set_v1` training artifacts, import the baseline estimators and evaluation utilities, and confirm that the baseline notebook is operating on the training split only.

No cross-validation or model fitting is performed yet. The purpose is to establish a stable, shared baseline-modeling context before the common evaluation protocol is introduced.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FEATURE_SET_VERSION = "feature_set_v1"
ARTIFACT_VERSION = "v1"
RANDOM_STATE = 42
TARGET_COLUMN = "is_canceled"

PROCESSED_DATA_CANDIDATES = [
    Path("../data/processed"),
    Path("data/processed"),
]
PROCESSED_DATA_DIR = next(
    (path for path in PROCESSED_DATA_CANDIDATES if path.exists()),
    PROCESSED_DATA_CANDIDATES[0],
)

if not PROCESSED_DATA_DIR.exists():
    raise FileNotFoundError("The processed data directory containing feature_set_v1 was not found.")

X_train_feat_path = PROCESSED_DATA_DIR / f"X_train_feat_{ARTIFACT_VERSION}.csv"
y_train_path = PROCESSED_DATA_DIR / f"y_train_{ARTIFACT_VERSION}.csv"

required_paths = [X_train_feat_path, y_train_path]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing expected Notebook 04 artifacts: {missing_paths}")

X_train_feat = pd.read_csv(X_train_feat_path)
y_train = pd.read_csv(y_train_path)[TARGET_COLUMN]

if len(X_train_feat) != len(y_train):
    raise ValueError("X_train_feat and y_train row counts do not match.")

baseline_setup_summary = pd.DataFrame(
    {
        "check": [
            "processed_data_dir",
            "feature_set_version",
            "X_train_feat_shape",
            "y_train_rows",
            "train_cancellation_rate_pct",
            "random_state",
        ],
        "value": [
            str(PROCESSED_DATA_DIR.resolve()),
            FEATURE_SET_VERSION,
            str(X_train_feat.shape),
            len(y_train),
            round(y_train.mean() * 100, 2),
            RANDOM_STATE,
        ],
    }
)

display(baseline_setup_summary)
display(X_train_feat.head())


,check,value
0,processed_data_dir,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,feature_set_version,feature_set_v1
2,X_train_feat_shape,"(69577, 44)"
3,y_train_rows,69577
4,train_cancellation_rate_pct,27.31
5,random_state,42


,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,adr_per_person,arrival_season,is_summer,log_lead_time,log_adr,log_adr_per_person,log_total_nights,country_grouped,has_agent,has_company
0,City Hotel,8,2016,October,41,6,0,1,1,0.0,...,65.00,autumn,0,2.197225,4.189655,4.189655,0.693147,PRT,0,1
1,Resort Hotel,208,2017,April,16,22,2,5,2,0.0,...,35.64,spring,0,5.342334,4.280547,3.601141,2.079442,IRL,1,0
2,Resort Hotel,0,2017,March,10,11,1,1,1,0.0,...,64.00,spring,0,0.000000,4.174387,4.174387,1.098612,Other,1,0
3,City Hotel,102,2017,June,26,30,1,2,2,0.0,...,67.50,summer,1,4.634729,4.912655,4.226834,1.386294,BEL,1,0
4,City Hotel,3,2016,July,29,11,1,0,1,0.0,...,65.00,summer,1,1.386294,4.189655,4.189655,0.693147,PRT,0,1


## Implementation Block 5.2 — Evaluation Protocol

**Scope.** Section 5.2 only — define the shared cross-validation protocol, identify numeric and categorical feature groups, and create reusable helpers so every baseline model is evaluated under the same rules.

No baseline model is run in this step. The purpose is to lock in one common evaluation procedure before the model roster is executed.

The protocol uses stratified 5-fold cross-validation with `random_state=42`, reports ROC-AUC as the primary metric, and keeps the default classification threshold at `0.5` for fold-level precision, recall, F1, and accuracy.


In [5]:
from sklearn.base import clone
from sklearn.impute import SimpleImputer

categorical_feature_columns = X_train_feat.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_feature_columns = [
    column for column in X_train_feat.columns if column not in categorical_feature_columns
]

cv_protocol = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
evaluation_metrics = ["roc_auc", "f1", "precision", "recall", "accuracy"]


def build_baseline_pipeline(estimator, scale_numeric):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler() if scale_numeric else "passthrough"),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_feature_columns),
            ("categorical", categorical_transformer, categorical_feature_columns),
        ]
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )


def evaluate_baseline_model(model_name, estimator, scale_numeric):
    fold_rows = []
    for fold_number, (train_index, valid_index) in enumerate(
        cv_protocol.split(X_train_feat, y_train),
        start=1,
    ):
        X_fold_train = X_train_feat.iloc[train_index]
        X_fold_valid = X_train_feat.iloc[valid_index]
        y_fold_train = y_train.iloc[train_index]
        y_fold_valid = y_train.iloc[valid_index]

        model_pipeline = build_baseline_pipeline(clone(estimator), scale_numeric=scale_numeric)
        model_pipeline.fit(X_fold_train, y_fold_train)

        validation_predictions = model_pipeline.predict(X_fold_valid)
        if hasattr(model_pipeline, "predict_proba"):
            validation_scores = model_pipeline.predict_proba(X_fold_valid)[:, 1]
        else:
            validation_scores = model_pipeline.decision_function(X_fold_valid)

        fold_rows.append(
            {
                "model_name": model_name,
                "fold": fold_number,
                "roc_auc": roc_auc_score(y_fold_valid, validation_scores),
                "f1": f1_score(y_fold_valid, validation_predictions),
                "precision": precision_score(y_fold_valid, validation_predictions, zero_division=0),
                "recall": recall_score(y_fold_valid, validation_predictions, zero_division=0),
                "accuracy": accuracy_score(y_fold_valid, validation_predictions),
            }
        )

    fold_results = pd.DataFrame(fold_rows)
    summary_results = fold_results[evaluation_metrics].agg(["mean", "std"]).T.reset_index()
    summary_results.columns = ["metric", "mean", "std"]
    summary_results.insert(0, "model_name", model_name)
    return fold_results, summary_results


cv_protocol_summary = pd.DataFrame(
    {
        "check": [
            "cv_strategy",
            "n_splits",
            "shuffle",
            "random_state",
            "numeric_feature_columns",
            "categorical_feature_columns",
            "numeric_imputer_strategy",
            "categorical_imputer_strategy",
            "primary_metric",
            "secondary_metrics",
        ],
        "value": [
            "StratifiedKFold",
            cv_protocol.get_n_splits(),
            True,
            RANDOM_STATE,
            len(numeric_feature_columns),
            len(categorical_feature_columns),
            "median",
            "most_frequent",
            "roc_auc",
            ", ".join([metric for metric in evaluation_metrics if metric != "roc_auc"]),
        ],
    }
)

display(cv_protocol_summary)


,check,value
0,cv_strategy,StratifiedKFold
1,n_splits,5
2,shuffle,True
3,random_state,42
4,numeric_feature_columns,32
5,categorical_feature_columns,12
6,numeric_imputer_strategy,median
7,categorical_imputer_strategy,most_frequent
8,primary_metric,roc_auc
9,secondary_metrics,"f1, precision, recall, accuracy"


## Implementation Block 5.3 — Baseline Model Roster

**Scope.** Section 5.3 only — define the untuned baseline models that will be compared under the shared evaluation protocol.

The roster intentionally mixes linear, tree-based, distance-based, and floor-baseline models. Each model is kept close to default settings so the comparison remains a true baseline rather than a tuning exercise.

The purpose of this block is to lock in the comparison roster before any model evaluation results are generated.


In [3]:
baseline_model_specs = [
    {
        "model_name": "Dummy Most Frequent",
        "estimator": DummyClassifier(strategy="most_frequent"),
        "scale_numeric": False,
        "family": "floor_baseline",
        "notes": "Mandatory majority-class reference baseline.",
    },
    {
        "model_name": "Logistic Regression",
        "estimator": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "scale_numeric": True,
        "family": "linear",
        "notes": "Scale-sensitive linear baseline with default decision threshold.",
    },
    {
        "model_name": "Decision Tree",
        "estimator": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "scale_numeric": False,
        "family": "tree",
        "notes": "Untuned single-tree baseline for interpretability and overfitting contrast.",
    },
    {
        "model_name": "Random Forest",
        "estimator": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
        "scale_numeric": False,
        "family": "tree_ensemble",
        "notes": "Modest ensemble baseline with default depth behavior.",
    },
    {
        "model_name": "K-Nearest Neighbors",
        "estimator": KNeighborsClassifier(),
        "scale_numeric": True,
        "family": "distance",
        "notes": "Scale-sensitive neighborhood baseline for contrast with linear and tree models.",
    },
]

baseline_roster_summary = pd.DataFrame(
    [
        {
            "model_name": spec["model_name"],
            "family": spec["family"],
            "scale_numeric": spec["scale_numeric"],
            "notes": spec["notes"],
        }
        for spec in baseline_model_specs
    ]
)

baseline_family_summary = (
    baseline_roster_summary.groupby("family", dropna=False)
    .size()
    .reset_index(name="model_count")
    .sort_values(by=["family"])
    .reset_index(drop=True)
)

display(baseline_roster_summary)
display(baseline_family_summary)


,model_name,family,scale_numeric,notes
0,Dummy Most Frequent,floor_baseline,False,Mandatory majority-class reference baseline.
1,Logistic Regression,linear,True,Scale-sensitive linear baseline with default d...
2,Decision Tree,tree,False,Untuned single-tree baseline for interpretabil...
3,Random Forest,tree_ensemble,False,Modest ensemble baseline with default depth be...
4,K-Nearest Neighbors,distance,True,Scale-sensitive neighborhood baseline for cont...


,family,model_count
0,distance,1
1,floor_baseline,1
2,linear,1
3,tree,1
4,tree_ensemble,1


## Implementation Block 5.4 — Run And Record

**Scope.** Section 5.4 only — evaluate each baseline model under the shared cross-validation protocol, collect fold-level metrics, and assemble both long-form and report-style comparison tables.

The purpose of this block is to generate a consistent baseline comparison table before any shortlist interpretation or artifact persistence is added.


In [6]:
baseline_fold_results_frames = []
baseline_summary_results_frames = []

for model_spec in baseline_model_specs:
    fold_results, summary_results = evaluate_baseline_model(
        model_name=model_spec["model_name"],
        estimator=model_spec["estimator"],
        scale_numeric=model_spec["scale_numeric"],
    )
    fold_results["family"] = model_spec["family"]
    fold_results["notes"] = model_spec["notes"]
    summary_results["family"] = model_spec["family"]
    summary_results["notes"] = model_spec["notes"]
    baseline_fold_results_frames.append(fold_results)
    baseline_summary_results_frames.append(summary_results)

baseline_fold_results = pd.concat(baseline_fold_results_frames, ignore_index=True)
baseline_results_long = pd.concat(baseline_summary_results_frames, ignore_index=True)

baseline_results_wide = (
    baseline_results_long.pivot(index="model_name", columns="metric", values="mean")
    .reset_index()
    .merge(
        baseline_roster_summary[["model_name", "family", "notes"]],
        on="model_name",
        how="left",
    )
    .sort_values(by=["roc_auc", "f1", "accuracy"], ascending=[False, False, False])
    .reset_index(drop=True)
)

baseline_run_summary = pd.DataFrame(
    {
        "check": [
            "models_evaluated",
            "fold_rows_recorded",
            "summary_rows_recorded",
            "top_model_by_roc_auc",
            "top_model_roc_auc",
        ],
        "value": [
            baseline_results_long["model_name"].nunique(),
            len(baseline_fold_results),
            len(baseline_results_long),
            baseline_results_wide.loc[0, "model_name"],
            round(float(baseline_results_wide.loc[0, "roc_auc"]), 4),
        ],
    }
)

display(baseline_results_long)
display(baseline_results_wide)
display(baseline_run_summary)


,model_name,metric,mean,std,family,notes
0,Dummy Most Frequent,roc_auc,0.500000,0.000000,floor_baseline,Mandatory majority-class reference baseline.
1,Dummy Most Frequent,f1,0.000000,0.000000,floor_baseline,Mandatory majority-class reference baseline.
2,Dummy Most Frequent,precision,0.000000,0.000000,floor_baseline,Mandatory majority-class reference baseline.
3,Dummy Most Frequent,recall,0.000000,0.000000,floor_baseline,Mandatory majority-class reference baseline.
4,Dummy Most Frequent,accuracy,0.726907,0.000027,floor_baseline,Mandatory majority-class reference baseline.
5,Logistic Regression,roc_auc,0.864123,0.002315,linear,Scale-sensitive linear baseline with default d...
6,Logistic Regression,f1,0.610623,0.000508,linear,Scale-sensitive linear baseline with default d...
7,Logistic Regression,precision,0.701488,0.009915,linear,Scale-sensitive linear baseline with default d...
8,Logistic Regression,recall,0.540708,0.005454,linear,Scale-sensitive linear baseline with default d...
9,Logistic Regression,accuracy,0.811676,0.002112,linear,Scale-sensitive linear baseline with default d...


,model_name,accuracy,f1,precision,recall,roc_auc,family,notes
0,Random Forest,0.846918,0.688566,0.774707,0.619704,0.901858,tree_ensemble,Modest ensemble baseline with default depth be...
1,Logistic Regression,0.811676,0.610623,0.701488,0.540708,0.864123,linear,Scale-sensitive linear baseline with default d...
2,K-Nearest Neighbors,0.796168,0.619576,0.631821,0.607810,0.825733,distance,Scale-sensitive neighborhood baseline for cont...
3,Decision Tree,0.794329,0.626032,0.621803,0.630388,0.743893,tree,Untuned single-tree baseline for interpretabil...
4,Dummy Most Frequent,0.726907,0.000000,0.000000,0.000000,0.500000,floor_baseline,Mandatory majority-class reference baseline.


,check,value
0,models_evaluated,5
1,fold_rows_recorded,25
2,summary_rows_recorded,25
3,top_model_by_roc_auc,Random Forest
4,top_model_roc_auc,0.9019


## Implementation Block 5.5 — Interpretation

**Scope.** Section 5.5 only — interpret the baseline comparison table, quantify which models meaningfully beat the majority-class floor, and produce a justified shortlist for Notebook 06.

The shortlist is based on a mix of performance, family diversity, and tuning tractability rather than ROC-AUC alone. The purpose is to make the promotion decision explicit before saving artifacts.


In [7]:
dummy_baseline_row = baseline_results_wide.loc[
    baseline_results_wide["model_name"] == "Dummy Most Frequent"
].iloc[0]

baseline_interpretation_summary = baseline_results_wide.copy()
baseline_interpretation_summary["roc_auc_uplift_vs_dummy"] = (
    baseline_interpretation_summary["roc_auc"] - float(dummy_baseline_row["roc_auc"])
).round(4)
baseline_interpretation_summary["f1_uplift_vs_dummy"] = (
    baseline_interpretation_summary["f1"] - float(dummy_baseline_row["f1"])
).round(4)
baseline_interpretation_summary["beats_dummy"] = baseline_interpretation_summary["roc_auc"] > float(
    dummy_baseline_row["roc_auc"]
)

shortlist_for_tuning = pd.DataFrame(
    [
        {
            "model_name": "Random Forest",
            "why_promoted": "Best ROC-AUC and strongest overall baseline performance, making it the leading ensemble candidate for tuning.",
            "tuning_priority": 1,
        },
        {
            "model_name": "Logistic Regression",
            "why_promoted": "Strong ROC-AUC with a transparent linear decision rule, giving an interpretable benchmark for tuned comparisons.",
            "tuning_priority": 2,
        },
        {
            "model_name": "K-Nearest Neighbors",
            "why_promoted": "Competitive ROC-AUC and F1 while representing a distinct distance-based family that may respond to hyperparameter tuning.",
            "tuning_priority": 3,
        },
    ]
)

shortlist_for_tuning = shortlist_for_tuning.merge(
    baseline_results_wide[["model_name", "roc_auc", "f1", "precision", "recall", "accuracy"]],
    on="model_name",
    how="left",
)

interpretation_checklist_summary = pd.DataFrame(
    {
        "check": [
            "models_beating_dummy_on_roc_auc",
            "shortlist_size",
            "top_shortlisted_model",
        ],
        "value": [
            int(baseline_interpretation_summary["beats_dummy"].sum()),
            len(shortlist_for_tuning),
            shortlist_for_tuning.sort_values("tuning_priority").iloc[0]["model_name"],
        ],
    }
)

display(baseline_interpretation_summary)
display(shortlist_for_tuning.sort_values("tuning_priority"))
display(interpretation_checklist_summary)


,model_name,accuracy,f1,precision,recall,roc_auc,family,notes,roc_auc_uplift_vs_dummy,f1_uplift_vs_dummy,beats_dummy
0,Random Forest,0.846918,0.688566,0.774707,0.619704,0.901858,tree_ensemble,Modest ensemble baseline with default depth be...,0.4019,0.6886,True
1,Logistic Regression,0.811676,0.610623,0.701488,0.540708,0.864123,linear,Scale-sensitive linear baseline with default d...,0.3641,0.6106,True
2,K-Nearest Neighbors,0.796168,0.619576,0.631821,0.607810,0.825733,distance,Scale-sensitive neighborhood baseline for cont...,0.3257,0.6196,True
3,Decision Tree,0.794329,0.626032,0.621803,0.630388,0.743893,tree,Untuned single-tree baseline for interpretabil...,0.2439,0.6260,True
4,Dummy Most Frequent,0.726907,0.000000,0.000000,0.000000,0.500000,floor_baseline,Mandatory majority-class reference baseline.,0.0000,0.0000,False


,model_name,why_promoted,tuning_priority,roc_auc,f1,precision,recall,accuracy
0,Random Forest,Best ROC-AUC and strongest overall baseline pe...,1,0.901858,0.688566,0.774707,0.619704,0.846918
1,Logistic Regression,Strong ROC-AUC with a transparent linear decis...,2,0.864123,0.610623,0.701488,0.540708,0.811676
2,K-Nearest Neighbors,Competitive ROC-AUC and F1 while representing ...,3,0.825733,0.619576,0.631821,0.607810,0.796168


,check,value
0,models_beating_dummy_on_roc_auc,4
1,shortlist_size,3
2,top_shortlisted_model,Random Forest


## Implementation Block 5.6 — Persist Artifacts

**Scope.** Section 5.6 only — save the baseline comparison table, generate the baseline ROC-AUC comparison figure, and export one experiment-log row per baseline model in a working log file.

The purpose of this block is to freeze the baseline outputs that Notebook 06 and the report will reuse.


In [8]:
import matplotlib.pyplot as plt

reports_tables_dir = Path("../reports/tables")
if not reports_tables_dir.exists():
    reports_tables_dir = Path("reports/tables")
reports_tables_dir.mkdir(parents=True, exist_ok=True)

reports_figures_dir = Path("../reports/figures")
if not reports_figures_dir.exists():
    reports_figures_dir = Path("reports/figures")
reports_figures_dir.mkdir(parents=True, exist_ok=True)

baseline_results_path = reports_tables_dir / "baseline_results_v1.csv"
baseline_figure_path = reports_figures_dir / "baseline_metric_comparison.png"
experiment_log_template_path = reports_tables_dir / "experiment_log_template.csv"
experiment_log_working_path = reports_tables_dir / "experiment_log_working.csv"

baseline_results_wide.to_csv(baseline_results_path, index=False)

plot_data = baseline_results_wide.sort_values("roc_auc", ascending=True)
figure, axis = plt.subplots(figsize=(10, 5))
axis.barh(plot_data["model_name"], plot_data["roc_auc"], color="#3b6fb6")
axis.set_title("Baseline Model Comparison by ROC-AUC")
axis.set_xlabel("Mean ROC-AUC")
axis.set_ylabel("Model")
axis.set_xlim(0, 1)
figure.tight_layout()
figure.savefig(baseline_figure_path, dpi=200, bbox_inches="tight")
plt.close(figure)

experiment_log_template = pd.read_csv(experiment_log_template_path)
experiment_log_rows = []
for row_number, baseline_row in enumerate(
    baseline_results_wide.sort_values("roc_auc", ascending=False).itertuples(index=False),
    start=1,
):
    experiment_log_rows.append(
        {
            "run_id": f"baseline_v1_{row_number:02d}",
            "run_date": "2026-04-19",
            "notebook_stage": "05_baseline_models",
            "model_name": baseline_row.model_name,
            "feature_set_version": FEATURE_SET_VERSION,
            "preprocessing_version": ARTIFACT_VERSION,
            "validation_strategy": "StratifiedKFold",
            "cv_folds": cv_protocol.get_n_splits(),
            "decision_threshold": 0.5,
            "random_state": RANDOM_STATE,
            "roc_auc": round(float(baseline_row.roc_auc), 6),
            "f1": round(float(baseline_row.f1), 6),
            "precision": round(float(baseline_row.precision), 6),
            "recall": round(float(baseline_row.recall), 6),
            "accuracy": round(float(baseline_row.accuracy), 6),
            "primary_metric": "roc_auc",
            "secondary_metrics": "f1;precision;recall;accuracy",
            "notes": baseline_row.notes,
        }
    )

experiment_log_working = pd.DataFrame(experiment_log_rows, columns=experiment_log_template.columns)
experiment_log_working.to_csv(experiment_log_working_path, index=False)

baseline_artifact_summary = pd.DataFrame(
    {
        "artifact": [
            "baseline_results_v1",
            "baseline_metric_comparison_figure",
            "baseline_experiment_log_working_copy",
        ],
        "path": [
            str(baseline_results_path.resolve()),
            str(baseline_figure_path.resolve()),
            str(experiment_log_working_path.resolve()),
        ],
    }
)

display(baseline_artifact_summary)


,artifact,path
0,baseline_results_v1,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,baseline_metric_comparison_figure,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
2,baseline_experiment_log_working_copy,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...


## Review Checklist

- [x] Every baseline uses the same `feature_set_v1` and the same 5-fold stratified CV protocol.
- [x] Majority-class dummy classifier is included as the floor.
- [x] At least three families are represented (linear, tree-based, distance-based).
- [x] `random_state=42` is used consistently.
- [x] All baselines are logged to the working experiment log with matching `run_id`s.
- [x] Results table saved to `reports/tables/baseline_results_v1.csv`.
- [x] ROC-AUC comparison figure saved to `reports/figures/baseline_metric_comparison.png`.
- [x] Shortlist justification is written in this notebook and considers more than ROC-AUC alone.
- [x] The held-out test set has not been touched.


## Handoff To Notebook 06

- Frozen baseline artifacts are ready: `baseline_results_v1.csv`, `baseline_metric_comparison.png`, and `experiment_log_working.csv`.
- The shortlisted models for tuning are Random Forest, Logistic Regression, and K-Nearest Neighbors.
- Notebook 06 should reuse the same preprocessing structure, `feature_set_v1`, primary metric (`roc_auc`), and 5-fold stratified CV protocol so tuned-vs-baseline comparisons stay fair.
- The held-out test set remains untouched and should stay untouched until notebook 07.
